# Predict

Run prediction using the model output from [02_modelling.ipynb](./02_modelling.ipynb)

Output: [nisjedata-substrat-xgbclassifier_norge_latest_25833.geojson](https://storage.googleapis.com/niva-geodata/MarintNaturKart/nisjedata-substrat-xgbclassifier_norge_latest_25833.geojson)

Comparison map [here](https://terriamap.p.niva.no/#start=%7B%22version%22%3A%228.0.0%22%2C%22initSources%22%3A%5B%7B%22stratum%22%3A%22user%22%2C%22models%22%3A%7B%22__User-Added_Data__%22%3A%7B%22isOpen%22%3Atrue%2C%22members%22%3A%5B%228245f6bc-c3d0-4fcc-aeaa-10faad0a3ec1%22%5D%2C%22knownContainerUniqueIds%22%3A%5B%22%2F%22%5D%2C%22type%22%3A%22group%22%7D%2C%22%2F%2FMarint+Natur+Kart%2FTest+Hard%2FL%C3%B8s%2FBlanding+Prediksjon+-+Norge%22%3A%7B%22splitDirection%22%3A1%2C%22knownContainerUniqueIds%22%3A%5B%22%2F%2FMarint+Natur+Kart%22%5D%2C%22type%22%3A%22wfs%22%7D%2C%22%2F%2FMarint+Natur+Kart%2FBunntypeklasser+for+Prediksjon+Norge%22%3A%7B%22splitDirection%22%3A-1%2C%22knownContainerUniqueIds%22%3A%5B%22%2F%2FMarint+Natur+Kart%22%5D%2C%22type%22%3A%22wfs%22%7D%2C%228245f6bc-c3d0-4fcc-aeaa-10faad0a3ec1%22%3A%7B%22splitSourceItemId%22%3A%22%2F%2FMarint+Natur+Kart%2FTest+Hard%2FL%C3%B8s%2FBlanding+Prediksjon+-+Norge%22%2C%22dereferenced%22%3A%7B%22name%22%3A%22Test+Hard%2FL%C3%B8s%2FBlanding+Prediksjon+-+Norge+%28copy%29%22%2C%22splitDirection%22%3A-1%7D%2C%22knownContainerUniqueIds%22%3A%5B%22__User-Added_Data__%22%5D%2C%22type%22%3A%22split-reference%22%7D%2C%22%2F%22%3A%7B%22type%22%3A%22group%22%7D%2C%22%2F%2FMarint+Natur+Kart%22%3A%7B%22knownContainerUniqueIds%22%3A%5B%22%2F%22%5D%2C%22type%22%3A%22group%22%7D%7D%2C%22workbench%22%3A%5B%22%2F%2FMarint+Natur+Kart%2FTest+Hard%2FL%C3%B8s%2FBlanding+Prediksjon+-+Norge%22%2C%22%2F%2FMarint+Natur+Kart%2FBunntypeklasser+for+Prediksjon+Norge%22%5D%2C%22timeline%22%3A%5B%22%2F%2FMarint+Natur+Kart%2FTest+Hard%2FL%C3%B8s%2FBlanding+Prediksjon+-+Norge%22%5D%2C%22initialCamera%22%3A%7B%22west%22%3A-1.8237304687500002%2C%22south%22%3A60.31062731740045%2C%22east%22%3A18.665771484375004%2C%22north%22%3A65.29346780107583%7D%2C%22homeCamera%22%3A%7B%22west%22%3A4%2C%22south%22%3A57.00000000000001%2C%22east%22%3A32%2C%22north%22%3A72%7D%2C%22viewerMode%22%3A%222d%22%2C%22showSplitter%22%3Atrue%2C%22splitPosition%22%3A0.5%2C%22settings%22%3A%7B%22baseMaximumScreenSpaceError%22%3A2%2C%22useNativeResolution%22%3Afalse%2C%22alwaysShowTimeline%22%3Afalse%2C%22baseMapId%22%3A%22basemap-openstreetmap%22%2C%22terrainSplitDirection%22%3A0%2C%22depthTestAgainstTerrainEnabled%22%3Afalse%7D%2C%22stories%22%3A%5B%5D%7D%5D%7D)

In [1]:
from pathlib import Path

import geopandas as gpd
import geoutils as gu
import numpy as np
import pandas as pd
import rasterio as rio
import xdem


import mnk.substrat as subkart
import mnk

In [2]:
res = subkart.features.RESOLUTION
nodata = 255
crs = "EPSG:25833"

In [3]:
classifier = subkart.utils.load_classifier()

# Predict for Norge

## Wave Exposure


In [4]:
bolge = mnk.sources.bolge_exposure()


## DEM 50

In [5]:
dem_norge = mnk.sources.dem_data()

## Predict for all of Norway in one pass

In [ ]:
print("Loading depth point data...")
gdf_points = mnk.sources.depth_point_data()
print(f"Depth points: {len(gdf_points):,}")


In [6]:
predict_file_unmapped = "predict_unmapped.tif"
prob_file = "3band_probability.tif"

print("Loading sea map for all of Norway...")
gdf_sea_map = mnk.sources.sea_map_basisdata()
gdf_sea_map = subkart.features.depth_preprocess(gdf_sea_map)
transform, out_shape, bounds = subkart.features.to_raster_shapes(gdf_sea_map, res=res)

print("Resampling DEM...")
dem = subkart.utils.resample_dem(dem_norge.crop(bounds), out_shape, transform, crs)

print("Reprojecting wave exposure...")
bolge_norge = bolge.reproject(
    crs=crs, res=res,
    bounds=dict(left=bounds[0], bottom=bounds[1], right=bounds[2], top=bounds[3]),
)

print("Building feature arrays...")
X, valid_attrs, out_shape, transform = subkart.features.build(
    dem, gdf_sea_map, bolge_norge, valid_mask=None, res=res, dtype=np.float32,
    gdf_points=gdf_points,
)

Loading sea map for all of Norway...
Reading preprocessed depth training data for all fylker.


All depth features already present, skipping preprocessing.
Resampling DEM...


Reprojecting wave exposure...


Building feature arrays...
Computing interpolated depth raster...


Preparing sea_avg_depth...


...

/home/jovyan/marint-naturkart-niva/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing sea_convexity...


/home/jovyan/marint-naturkart-niva/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing wave exposure...


Stacking feature arrays...


In [7]:
Y_pred = classifier.predict(X)
Y_prob = classifier.predict_proba(X)

# Prediction raster (classes 0=løsbunn, 1=blanding, 2=fastbunn)
pred_map = np.full(out_shape, nodata, dtype=np.uint8)
pred_map[valid_attrs] = Y_pred.astype(np.uint8)
pred_map_masked = np.ma.masked_equal(pred_map, nodata)
pred_raster = gu.Raster.from_array(pred_map_masked, transform=transform, crs=crs, nodata=nodata)
pred_raster.to_file(predict_file_unmapped)
print(f"Saved {predict_file_unmapped}")

# Three-band probability raster (band 1=P(løsbunn), band 2=P(blanding), band 3=P(fastbunn))
prob_nodata = np.float32(-9999)
prob_bands = np.full((len(classifier.classes_), *out_shape), prob_nodata, dtype=np.float32)
for band_idx in range(len(classifier.classes_)):
    prob_bands[band_idx][valid_attrs] = Y_prob[:, band_idx].astype(np.float32)
with rio.open(
    prob_file, "w",
    driver="GTiff",
    height=out_shape[0], width=out_shape[1],
    count=len(classifier.classes_),
    dtype=np.float32,
    crs=crs,
    transform=transform,
    nodata=prob_nodata,
    compress="deflate",
    tiled=True,
    blockxsize=512,
    blockysize=512,
) as dst:
    dst.write(prob_bands)
print(f"Saved {prob_file}")

Saved predict_unmapped.tif


Saved 3band_probability.tif
